In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

In [2]:
# ── Same tiny dataset ─────────────────────────────────────────────────
X = torch.randn(10, 4)
y = torch.randint(0, 2, (10,))
X_train, y_train = X[:8], y[:8]
X_val,   y_val   = X[8:], y[8:]

In [3]:
# ── Model with BatchNorm + Dropout ────────────────────────────────────
class FullModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128),
            nn.BatchNorm1d(128),   # Normalize after linear, before activation
            nn.ReLU(),
            nn.Dropout(p=0.4),     # Then dropout

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),    # Again in second layer
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(64, 2)       # Final output layer — no BN or Dropout here
        )

    def forward(self, x):
        return self.net(x)

In [4]:
model = FullModel()

In [5]:
# ── Optimizer with L2 ─────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()

In [6]:
# ── Training loop ─────────────────────────────────────────────────────
for epoch in range(200):
    model.train()                  # BN + Dropout both active in train mode
    optimizer.zero_grad()
    out = model(X_train)
    train_loss = criterion(out, y_train)
    train_loss.backward()
    optimizer.step()

    model.eval()                   # BN + Dropout both disabled in eval mode
    with torch.no_grad():
        val_out = model(X_val)
        val_loss = criterion(val_out, y_val)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

Epoch  10 | Train Loss: 0.1121 | Val Loss: 1.3865
Epoch  20 | Train Loss: 0.0115 | Val Loss: 2.0997
Epoch  30 | Train Loss: 0.0021 | Val Loss: 2.4809
Epoch  40 | Train Loss: 0.0052 | Val Loss: 2.7426
Epoch  50 | Train Loss: 0.0015 | Val Loss: 2.7860
Epoch  60 | Train Loss: 0.0161 | Val Loss: 2.6940
Epoch  70 | Train Loss: 0.0015 | Val Loss: 2.7788
Epoch  80 | Train Loss: 0.0005 | Val Loss: 2.6623
Epoch  90 | Train Loss: 0.0014 | Val Loss: 2.8048
Epoch 100 | Train Loss: 0.0047 | Val Loss: 2.9331
Epoch 110 | Train Loss: 0.0009 | Val Loss: 2.7469
Epoch 120 | Train Loss: 0.0093 | Val Loss: 2.6561
Epoch 130 | Train Loss: 0.0025 | Val Loss: 2.6726
Epoch 140 | Train Loss: 0.0012 | Val Loss: 2.6607
Epoch 150 | Train Loss: 0.0061 | Val Loss: 2.7546
Epoch 160 | Train Loss: 0.0010 | Val Loss: 2.9843
Epoch 170 | Train Loss: 0.0015 | Val Loss: 3.1260
Epoch 180 | Train Loss: 0.0027 | Val Loss: 3.0012
Epoch 190 | Train Loss: 0.0058 | Val Loss: 2.8005
Epoch 200 | Train Loss: 0.0018 | Val Loss: 2.9707
